# Rare Disease Genes – Homework 3: Visualization

**Instructions:** Run the Setup cell first, then complete each exercise in order.  
Write your code in the `# YOUR CODE HERE` cells. Answer written questions in the markdown cells below each prompt.

**What you'll learn:**
- Merge two datasets and explore the combined data
- Create bar charts, pie charts, and scatter plots
- Use histograms and box plots to explore distributions
- Build a heatmap to spot patterns across two dimensions
- Compare groups using grouped bar charts
- Investigate disease–gene patterns across chromosomes

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load both datasets
diseases = pd.read_csv('https://raw.githubusercontent.com/narayananr/rare_diseases/main/data/rare_gene_disease_dataset.csv')
genes = pd.read_csv('https://raw.githubusercontent.com/narayananr/rare_diseases/main/data/ensembl_genes.csv')

# Merge them (you learned this in Class 2!)
merged = diseases.merge(genes, on='gene_symbol', how='left')

# Calculate gene length (we'll use this throughout)
genes['length'] = genes['end'] - genes['start']
merged['length'] = merged['end'] - merged['start']

print(f'Diseases dataset: {len(diseases)} rows')
print(f'Genes dataset: {len(genes)} rows')
print(f'Merged dataset: {len(merged)} rows, {merged.shape[1]} columns')
print(f'\nColumns: {list(merged.columns)}')
merged.head()

---
## Part 1: Rare Disease Genes by Chromosome

Now that we have the merged dataset, we can answer questions that neither dataset could answer alone.  
Let’s start: **which chromosomes carry the most rare disease genes?**

In [ ]:
# Demo: Count rare disease genes per chromosome (sorted in chromosome order)
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT']

disease_by_chrom = merged['chromosome'].value_counts().reindex(chrom_order).dropna()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(disease_by_chrom.index, disease_by_chrom.values, color='steelblue')
ax.set_xlabel('Chromosome')
ax.set_ylabel('Number of Disease-Gene Links')
ax.set_title('Rare Disease-Gene Links per Chromosome')
plt.tight_layout()
plt.show()

### 🎯 Exercise 1: Unique Disease Genes per Chromosome

The chart above counts disease-gene **links** (one gene can appear many times if it's linked to many diseases).  
Now create a chart that counts **unique genes** per chromosome instead.

*Hints:*
- Use `merged.drop_duplicates(subset='gene_symbol')` to get one row per gene
- Then count by chromosome with `.value_counts()` and `.reindex(chrom_order)`
- Use the demo chart above as a template

In [ ]:
# YOUR CODE HERE: Bar chart of unique rare disease genes per chromosome


**Answer these questions** (double-click to edit):

1. Which chromosome has the most unique rare disease genes?  
   *Your answer:*

2. Does chromosome size seem to matter? (Chromosome 1 is the largest, chromosome 21 is one of the smallest.)  
   *Your answer:*

3. How does the X chromosome compare to similarly-sized autosomes?  
   *Your answer:*

---
## Part 2: Pie Charts — Gene Biotypes

A **pie chart** shows how a whole breaks into parts. It works best with a small number of categories.

The `biotype` column tells us what type of gene each one is:  
`protein_coding`, `lncRNA`, `miRNA`, etc.

Let’s see what types of genes are linked to rare diseases.

In [ ]:
# Demo: Pie chart of biotypes for rare disease genes
unique_disease_genes = merged.drop_duplicates(subset='gene_symbol')
biotype_counts = unique_disease_genes['biotype'].value_counts()

# Group small categories into "Other" so the chart is readable
top_biotypes = biotype_counts.head(5)
other_count = biotype_counts.iloc[5:].sum()
plot_data = pd.concat([top_biotypes, pd.Series({'Other': other_count})])

fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(plot_data.values, labels=plot_data.index, autopct='%1.1f%%', startangle=90)
ax.set_title('Biotypes of Rare Disease Genes')
plt.tight_layout()
plt.show()

print(f"\nBreakdown:")
for name, count in plot_data.items():
    print(f"  {name}: {count}")

### Pie Chart Quick Reference

| Line | What it does |
|------|--------------|
| `ax.pie(values, labels=names)` | Create a pie chart |
| `autopct='%1.1f%%'` | Show percentages on each slice |
| `startangle=90` | Rotate so first slice starts at top |

### 🎯 Exercise 2: Compare Biotypes — Rare Disease Genes vs All Genes

Create a pie chart for **all genes** in the Ensembl dataset (not just rare disease genes).  
Compare it to the rare disease pie chart above.

*Hints:*
- Use the `genes` dataframe (the full Ensembl dataset)
- Count biotypes with `genes['biotype'].value_counts()`
- Group small categories into "Other" (same as the demo above)
- Copy the pie chart code and change the data and title

In [ ]:
# YOUR CODE HERE: Pie chart of biotypes for ALL genes in Ensembl


**Answer these questions** (double-click to edit):

1. What percentage of rare disease genes are protein-coding? What about all genes?  
   *Your answer:*

2. Why do you think rare disease genes are overwhelmingly protein-coding?  
   *Your answer:*

---
## Part 3: Gene Length Comparison

Each gene has a `start` and `end` position on its chromosome.  
**Gene length** = `end` − `start` (measured in base pairs).

Are rare disease genes longer or shorter than the average gene?

In [ ]:
# Gene length is already calculated in the setup cell
unique_disease_genes = merged.drop_duplicates(subset='gene_symbol').copy()

# Compare averages
avg_all = genes['length'].mean()
avg_rare = unique_disease_genes['length'].dropna().mean()

print(f"Average gene length (all genes):          {avg_all:,.0f} base pairs")
print(f"Average gene length (rare disease genes):  {avg_rare:,.0f} base pairs")

### 🎯 Exercise 3: Grouped Bar Chart — Gene Length by Chromosome

A **grouped bar chart** places two bars side by side for each category, making it easy to compare.  
Create a grouped bar chart comparing **average gene length** on each chromosome for:
- All genes
- Rare disease genes

Use the template below — fill in the missing parts.

In [ ]:
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

# Average gene length per chromosome for all genes
avg_all_by_chrom = genes.groupby('chromosome')['length'].mean().reindex(chrom_order)

# YOUR CODE HERE: Average gene length per chromosome for rare disease genes
# Hint: use unique_disease_genes.groupby('chromosome')['length'].mean()
# avg_rare_by_chrom = ???

# Grouped bar chart
x = np.arange(len(chrom_order))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width/2, avg_all_by_chrom.values / 1000, width, label='All Genes', color='lightcoral')
# YOUR CODE HERE: add the second set of bars for rare disease genes
# ax.bar(x + width/2, ???, width, label='Rare Disease Genes', color='steelblue')

ax.set_xlabel('Chromosome')
ax.set_ylabel('Average Gene Length (kb)')
ax.set_title('Average Gene Length: All Genes vs Rare Disease Genes')
ax.set_xticks(x)
ax.set_xticklabels(chrom_order)
ax.legend()
plt.tight_layout()
plt.show()

### Grouped Bar Chart Quick Reference

| Line | What it does |
|------|--------------|
| `x = np.arange(len(categories))` | Create evenly spaced positions |
| `ax.bar(x - width/2, values, width)` | Left group of bars |
| `ax.bar(x + width/2, values, width)` | Right group of bars |
| `ax.legend()` | Show which color is which group |
| `/ 1000` | Convert base pairs to kilobases (kb) |

**Answer this question** (double-click to edit):

1. Are rare disease genes generally longer or shorter than the average gene? Why might that be?  
   *Your answer:*

---
## Part 4: Scatter Plots

A **scatter plot** shows the relationship between two numbers.  
Each dot is one data point — its position on the x-axis shows one value, and its position on the y-axis shows another.

**Question:** Do chromosomes with more total genes also have more rare disease genes?

In [ ]:
# Demo: Scatter plot — total genes vs rare disease genes per chromosome
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

# Count total genes per chromosome
total_genes = genes[genes['chromosome'].isin(chrom_order)].groupby('chromosome').size()
total_genes = total_genes.reindex(chrom_order)

# Count unique rare disease genes per chromosome
disease_genes = merged.drop_duplicates(subset='gene_symbol')
rare_genes = disease_genes[disease_genes['chromosome'].isin(chrom_order)].groupby('chromosome').size()
rare_genes = rare_genes.reindex(chrom_order).fillna(0)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(total_genes.values, rare_genes.values, color='steelblue', s=80)

# Label each dot with the chromosome name
for chrom in chrom_order:
    ax.annotate(chrom, (total_genes[chrom], rare_genes[chrom]),
                textcoords='offset points', xytext=(5, 5), fontsize=9)

ax.set_xlabel('Total Genes on Chromosome')
ax.set_ylabel('Rare Disease Genes on Chromosome')
ax.set_title('Total Genes vs Rare Disease Genes per Chromosome')
plt.tight_layout()
plt.show()

### Scatter Plot Quick Reference

| Line | What it does |
|------|--------------|
| `ax.scatter(x, y)` | Create a scatter plot |
| `s=80` | Dot size |
| `ax.annotate(text, (x, y))` | Label a dot |
| `xytext=(5, 5)` | Offset the label so it doesn’t sit on the dot |

### 🎯 Exercise 4: Interpret the Scatter Plot

Look at the scatter plot above and answer these questions (double-click to edit):

1. Is there a relationship between total genes and rare disease genes? Describe the pattern.  
   *Your answer:*

2. Which chromosome seems to have **more** rare disease genes than you’d expect based on its total gene count? (Look for dots that are higher than the trend.)  
   *Your answer:*

3. Which chromosome has **fewer** rare disease genes than expected?  
   *Your answer:*

4. Why might chromosome X be unusual? (Hint: think about X-linked genetic diseases.)  
   *Your answer:*

---
## Part 5: Disease Category Chromosome Hotspots

Different chromosomes may be hotspots for different types of diseases.  
Let’s find out which chromosomes carry the most genes for a disease category.

In [ ]:
# Demo: Which chromosomes have the most "cancer" genes?
keyword = 'cancer'
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

category = merged[merged['disease'].str.contains(keyword, case=False)]
category_genes = category.drop_duplicates(subset='gene_symbol')
chrom_counts = category_genes['chromosome'].value_counts().reindex(chrom_order).dropna()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(chrom_counts.index, chrom_counts.values, color='indianred')
ax.set_xlabel('Chromosome')
ax.set_ylabel(f'Unique Genes linked to "{keyword}" diseases')
ax.set_title(f'Rare Disease Genes for "{keyword}" by Chromosome')
plt.tight_layout()
plt.show()

print(f"\nTotal unique '{keyword}' disease genes: {len(category_genes)}")
print(f"Top 3 chromosomes: {', '.join(chrom_counts.head(3).index)}")

### 🎯 Exercise 5: Your Disease Category

Pick a **different keyword** from the table below (or your own) and create the same type of chart.

| Body system | Keywords | Condition types | Keywords |
|-------------|----------|-----------------|----------|
| Kidney | `kidney`, `renal` | Cancer | `cancer`, `tumor` |
| Liver | `liver`, `hepatic` | Muscle | `dystrophy`, `myopathy` |
| Eye | `eye`, `retinal` | Brain | `epilepsy`, `ataxia` |
| Heart | `heart`, `cardiac` | Blood | `anemia`, `thrombosis` |
| Lung | `lung`, `pulmonary` | Hearing | `deafness`, `hearing` |

In [ ]:
# YOUR CODE HERE: Pick a keyword and create a chromosome bar chart
my_keyword = ''   # Fill in your keyword

if not my_keyword:
    raise ValueError("Please fill in your keyword before running this cell!")

# Filter, deduplicate, count by chromosome, and plot


**Answer these questions** (double-click to edit):

1. Which keyword did you choose?  
   *Your answer:*

2. Which chromosome has the most genes for your category?  
   *Your answer:*

3. Are the results different from the cancer example? What might explain the differences?  
   *Your answer:*

---
## Part 6: Histograms — Gene Length Distribution

A **histogram** shows how values are spread out (their *distribution*).  
Instead of one bar per category, a histogram groups numbers into bins and counts how many fall in each bin.

**Question:** What do rare disease gene lengths look like — are most genes short, long, or somewhere in between?

In [ ]:
# Demo: Histogram comparing gene lengths — all genes vs rare disease genes
unique_disease_genes = merged.drop_duplicates(subset='gene_symbol')

fig, ax = plt.subplots(figsize=(10, 6))

# Plot both distributions on the same chart
sns.histplot(genes['length'] / 1000, bins=50, color='lightcoral', label='All Genes',
             alpha=0.5, stat='density', ax=ax)
sns.histplot(unique_disease_genes['length'].dropna() / 1000, bins=50, color='steelblue',
             label='Rare Disease Genes', alpha=0.5, stat='density', ax=ax)

ax.set_xlabel('Gene Length (kb)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Gene Lengths: All Genes vs Rare Disease Genes')
ax.set_xlim(0, 500)   # Zoom in — most genes are under 500 kb
ax.legend()
plt.tight_layout()
plt.show()

### Histogram Quick Reference

| Line | What it does |
|------|--------------|
| `sns.histplot(data, bins=50)` | Create a histogram with 50 bins |
| `alpha=0.5` | Make bars semi-transparent (so overlapping distributions are visible) |
| `stat='density'` | Normalize so the area under each curve sums to 1 (fair comparison when group sizes differ) |
| `ax.set_xlim(0, 500)` | Zoom in on x-axis range |

### 🎯 Exercise 6: Histogram for a Disease Category

Pick a disease keyword and create a histogram of gene lengths for that category.  
Overlay it with the "all genes" distribution so you can compare.

*Hint: Filter `merged` for your keyword, deduplicate by gene, then use `sns.histplot()` twice.*

In [ ]:
# YOUR CODE HERE: Histogram of gene lengths for a disease category


**Answer these questions** (double-click to edit):

1. Are most rare disease genes short or long?  
   *Your answer:*

2. Does the distribution for your disease category look different from the overall distribution?  
   *Your answer:*

---
## Part 7: Box Plots — Comparing Distributions

A **box plot** summarizes a distribution in one shape:
- The **box** spans the middle 50% of values (25th to 75th percentile)
- The **line inside** the box is the **median** (middle value)
- The **whiskers** extend to the rest of the data
- **Dots** beyond the whiskers are **outliers** (unusually extreme values)

Box plots are great for comparing distributions across groups side by side.

In [ ]:
# Demo: Box plot of gene length by biotype (top 5 biotypes only)
unique_disease_genes = merged.drop_duplicates(subset='gene_symbol').copy()
top5_biotypes = unique_disease_genes['biotype'].value_counts().head(5).index

plot_data = unique_disease_genes[unique_disease_genes['biotype'].isin(top5_biotypes)]

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=plot_data, x='biotype', y='length', order=top5_biotypes, ax=ax)
ax.set_ylabel('Gene Length (base pairs)')
ax.set_xlabel('Biotype')
ax.set_title('Gene Length Distribution by Biotype (Rare Disease Genes)')
ax.set_yscale('log')   # Log scale because gene lengths vary enormously
plt.tight_layout()
plt.show()

### Box Plot Quick Reference

| Line | What it does |
|------|--------------|
| `sns.boxplot(data=df, x='col1', y='col2')` | Create a box plot grouped by `x` |
| `order=[...]` | Control the order of categories on x-axis |
| `ax.set_yscale('log')` | Use log scale (helpful when values span many orders of magnitude) |

### 🎯 Exercise 7: Box Plot — Gene Length by Chromosome

Create a box plot showing **gene length distributions across chromosomes** for rare disease genes.

*Hints:*
- Use `unique_disease_genes` (already created above)
- Set `x='chromosome'` and `y='length'`
- Use `order=chrom_order` to sort chromosomes logically
- Use `ax.set_yscale('log')` since gene lengths vary a lot
- Make the figure wide: `figsize=(14, 6)`

In [ ]:
# YOUR CODE HERE: Box plot of gene length by chromosome


**Answer these questions** (double-click to edit):

1. Which chromosome has the widest spread of gene lengths? Which has the narrowest?  
   *Your answer:*

2. Are there many outliers (dots)? What does an outlier mean in this context?  
   *Your answer:*

---
## Part 8: Heatmaps — Two Dimensions at Once

A **heatmap** uses color to show values in a grid. Rows and columns represent categories, and the color intensity shows the count or value in each cell.

**Question:** Which disease categories cluster on which chromosomes?

In [ ]:
# Demo: Heatmap — disease categories by chromosome
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']

categories = {
    'Cancer': 'cancer|tumor|carcinoma|lymphoma|leukemia',
    'Heart': 'heart|cardiac|cardiomyopathy',
    'Brain': 'brain|neuro|epilepsy|ataxia|neuropathy',
    'Eye': 'eye|retinal|blindness|optic|macular',
    'Muscle': 'muscle|muscular|myopathy|dystrophy',
    'Kidney': 'kidney|renal|nephro',
    'Immune': 'immune|immunodeficiency',
    'Bone': 'bone|osteo|skeletal',
}

# Build a table: rows = categories, columns = chromosomes
heatmap_data = {}
for cat, pattern in categories.items():
    filtered = merged[merged['disease'].str.contains(pattern, case=False)]
    gene_counts = filtered.drop_duplicates(subset='gene_symbol')
    counts = gene_counts['chromosome'].value_counts().reindex(chrom_order).fillna(0)
    heatmap_data[cat] = counts

heatmap_df = pd.DataFrame(heatmap_data).T   # categories as rows

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_df, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_xlabel('Chromosome')
ax.set_ylabel('Disease Category')
ax.set_title('Rare Disease Genes: Disease Categories by Chromosome')
plt.tight_layout()
plt.show()

### Heatmap Quick Reference

| Line | What it does |
|------|--------------|
| `sns.heatmap(df)` | Create a heatmap from a DataFrame |
| `annot=True` | Show numbers in each cell |
| `fmt='.0f'` | Format numbers as integers (no decimals) |
| `cmap='YlOrRd'` | Color scheme (yellow → orange → red). Try `'Blues'`, `'Greens'`, `'viridis'` |

### 🎯 Exercise 8: Interpret the Heatmap

Look at the heatmap above and answer these questions (double-click to edit):

1. Which chromosome is a "hotspot" for cancer-related genes?  
   *Your answer:*

2. Which disease category has the most genes on chromosome X?  
   *Your answer:*

3. Are there any chromosomes that have very few disease genes across all categories? Which ones?  
   *Your answer:*

4. Pick one bright cell (high value) in the heatmap. Why might that chromosome have many genes for that disease type?  
   *Your answer:*

---
## Part 9: Your Own Investigation

Combine what you've learned to answer a question **you** find interesting using the merged dataset.

### 🎯 Exercise 9: Investigate and Visualize

Pick **one** question below (or make up your own) and answer it with code and a chart:

1. **Which chromosome has the longest rare disease genes?**  
   → Group by chromosome, calculate average gene length, make a bar chart

2. **Do forward-strand and reverse-strand genes have different disease counts?**  
   → Filter by strand (1 or -1), compare disease counts with a grouped bar or pie chart

3. **Compare two disease keywords — which chromosomes differ?**  
   → Filter merged for two keywords, count by chromosome, plot side by side

4. **Your own question!**  
   → Any question you can answer with the merged dataset

In [ ]:
# YOUR CODE HERE: Investigate your chosen question


In [ ]:
# YOUR CODE HERE: Create a visualization for your question


### 🎯 Exercise 10: Written Summary

In 3–5 sentences, summarize what you found in Exercise 9.  
(Double-click to edit)

- What question did you pick and why?
- What did your chart show?
- What was the most surprising finding?

*Your answer:*

---
## ⭐ Extension Challenge (Optional)

### Stacked Bar Chart: Multiple Disease Categories by Chromosome

Create a **stacked bar chart** showing how many genes from different disease categories are on each chromosome.  
Pick 3–4 keywords and stack them.

*Hint: Use `ax.bar()` multiple times with the `bottom` parameter to stack bars.*

In [ ]:
# YOUR CODE HERE: Stacked bar chart of disease categories by chromosome
#
# Template:
#   chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']
#   keywords = ['cancer', 'cardiac', 'renal']   # pick your own
#
#   fig, ax = plt.subplots(figsize=(14, 6))
#   bottom = np.zeros(len(chrom_order))
#
#   for kw in keywords:
#       filtered = merged[merged['disease'].str.contains(kw, case=False)]
#       counts = filtered.drop_duplicates(subset='gene_symbol')['chromosome'].value_counts()
#       counts = counts.reindex(chrom_order).fillna(0).values
#       ax.bar(chrom_order, counts, bottom=bottom, label=kw)
#       bottom += counts
#
#   ax.set_xlabel('Chromosome')
#   ax.set_ylabel('Unique Disease Genes')
#   ax.set_title('Disease Categories by Chromosome')
#   ax.legend()
#   plt.tight_layout()
#   plt.show()


---
## Quick Reference: All Chart Types

| Chart | When to use | Key function |
|-------|------------|-------------|
| Bar chart | Compare counts across categories | `ax.bar(x, y)` or `ax.barh(x, y)` |
| Pie chart | Show parts of a whole | `ax.pie(values, labels=names, autopct='%1.1f%%')` |
| Scatter plot | Show relationship between two numbers | `ax.scatter(x, y)` |
| Grouped bar | Compare two groups side by side | `ax.bar(x - w/2, ...)` + `ax.bar(x + w/2, ...)` |
| Histogram | Show distribution of values | `sns.histplot(data, bins=50)` |
| Box plot | Compare distributions across groups | `sns.boxplot(data=df, x='col1', y='col2')` |
| Heatmap | Show values across two dimensions | `sns.heatmap(df, annot=True, cmap='YlOrRd')` |
| Stacked bar | Show composition across categories | `ax.bar(x, y, bottom=prev)` |

### Common Styling

| Line | What it does |
|------|--------------|
| `fig, ax = plt.subplots(figsize=(10, 6))` | Set figure size |
| `ax.set_xlabel('...')` | Label x-axis |
| `ax.set_ylabel('...')` | Label y-axis |
| `ax.set_title('...')` | Add title |
| `ax.legend()` | Show legend |
| `ax.set_yscale('log')` | Log scale for large value ranges |
| `alpha=0.5` | Semi-transparent (for overlapping plots) |
| `plt.tight_layout()` | Prevent labels from being cut off |
| `plt.show()` | Display the chart |